# **Curso de Regulación y Ética en IA**
---
<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://github.com/AntonioEscamilla/images-in-readMe/raw/master/Curso%20IA/upb%20logo.jpg", align="left", width="350">
</div>
</div>

### **Facultad en Tecnologías de la Información y la Comunicación**
### Universidad Pontificia Bolivariana — 2026

# Proyecto Final: Mitigación de sesgo con *Exponentiated Gradient Reduction*

**Técnica asignada:** Exponentiated Gradient Reduction — Agarwal, Beygelzimer, Dudík, Langford y Wallach (2018),
*A Reductions Approach to Fair Classification*, ICML 2018, PMLR 80:60-69.

**Integrantes:** Luis Seoanes

**Contenido**

1. Decisiones de la Semana 1: framework, dataset y métricas
2. Fundamento teórico de la técnica
3. Carga y análisis exploratorio (EDA)
4. Preprocesamiento y partición train / validación / test
5. Modelo *baseline* sin mitigación
6. Mitigación con Exponentiated Gradient: barrido de hiperparámetros
7. Evaluación final en test (antes vs. después)
8. Análisis de sensibilidad y robustez
9. Comparación con una alternativa de post-procesamiento
10. Múltiples formas de discriminación: raza, interseccionalidad (sexo × raza) y edad
11. Conclusiones

# 1. Decisiones de la Semana 1

## 1.1 Framework: Fairlearn vs. AIF360

| Criterio | **Fairlearn** | **AIF360** (IBM) |
|---|---|---|
| Clase que implementa la técnica | `fairlearn.reductions.ExponentiatedGradient` | `aif360.algorithms.inprocessing.ExponentiatedGradientReduction` |
| Origen de la implementación | Implementación de referencia; mantenida por Microsoft Research y coautores del paper (M. Dudík) | *Wrapper* que importa y llama internamente la implementación de Fairlearn |
| Restricciones disponibles | `DemographicParity`, `EqualizedOdds`, `TruePositiveRateParity`, `FalsePositiveRateParity`, `ErrorRateParity`, `BoundedGroupLoss` | Las mismas (se pasan como texto a Fairlearn) |
| API | scikit-learn nativa (`fit`/`predict`, arrays o DataFrames) | Requiere convertir los datos a `BinaryLabelDataset` (o usar su API `aif360.sklearn`) |
| Métricas | `MetricFrame` + funciones de disparidad | Muy amplio catálogo (`ClassificationMetric`) |
| Dependencias / instalación | Ligero; funciona en Python 3.13 | Más pesado; dependencias opcionales por algoritmo |
| Licencia | MIT | Apache 2.0 |

**Decisión: Fairlearn.** AIF360 no añade nada al algoritmo (delega en Fairlearn), así que usar Fairlearn
directamente evita una capa de conversión de datos, es la implementación más cercana al paper y es el
framework que ya se usó en el curso (notebook 08). Las métricas también se calculan con Fairlearn
(`MetricFrame`) y se verifican a mano con la matriz de confusión.

## 1.2 Dataset: Adult Census Income

| Dataset | Binario | Atributo protegido | ¿Usado en el paper de Agarwal et al.? | Disponibilidad / documentación |
|---|---|---|---|---|
| **Adult Income** | `>50K` vs `<=50K` | sexo, raza | **Sí** (con sexo como atributo protegido) | UCI / OpenML, `fairlearn.datasets.fetch_adult`; 48.842 filas; datasheet conocido |
| COMPAS | reincide sí/no | raza | Sí | ProPublica; ~7.000 filas; problemas de calidad de la etiqueta documentados |
| German Credit | buen/mal crédito | sexo, edad | No | 1.000 filas: varianza alta en métricas por grupo |
| Hepatitis | sobrevive / fallece | sexo | No | 155 filas: muy pocas mujeres para estimar tasas por grupo |

**Decisión: Adult, con varios atributos protegidos:**

- **Sexo** (`sex`): atributo principal, el mismo que usa el paper. Las secciones 5 a 9 desarrollan todo el
  experimento sobre él.
- **Raza** (`race`) e **intersección sexo × raza**: se auditan y se mitigan en la sección 10.
- **Edad** (`age`): se audita en la sección 10 (protegida por la ADEA en EE. UU. y por la Directiva 2000/78/CE en la UE).

Razones de la elección:

- Es uno de los cuatro datasets del experimento original, lo que permite contrastar nuestros resultados
  con los del paper.
- Es binario de forma nativa: **no hay que binarizar** la variable objetivo, así que no se introduce un
  umbral arbitrario. (Sí hay un umbral implícito, 50.000 USD de 1994, definido por los creadores del dataset.)
- Tiene tamaño suficiente (~14.600 mujeres) para estimar tasas por grupo con poca varianza.
- Está documentado (Kohavi y Becker, 1996) y se descarga reproduciblemente desde OpenML.
- **Contexto de uso:** predecir ingreso alto es un *proxy* de decisiones de acceso a crédito, vivienda o
  empleo (p. ej. preselección de candidatos o límite de una tarjeta). Ese es el escenario que asumimos para
  el análisis ético-regulatorio.

## 1.3 Métricas de evaluación

El paper formula la equidad como restricciones lineales sobre momentos condicionales y evalúa con
**dos nociones de equidad** frente al **error de clasificación**:

| Métrica | Definición | Por qué la usamos |
|---|---|---|
| **Demographic Parity Difference (DPD)** | $\max_a P(\hat Y=1\mid A=a)-\min_a P(\hat Y=1\mid A=a)$ | Restricción *DP* del paper. Mide si el beneficio (ser clasificado `>50K`) se reparte igual entre sexos |
| **Demographic Parity Ratio (DPR)** | $\min_a P(\hat Y=1\mid A=a)\,/\,\max_a P(\hat Y=1\mid A=a)$ | Equivale a la *regla de los 4/5* de la EEOC (impacto dispar en empleo en EE. UU.): DPR ≥ 0.8 |
| **Equalized Odds Difference (EOD)** | $\max\big(\Delta\text{TPR},\ \Delta\text{FPR}\big)$ entre grupos | Restricción *EO* del paper (Hardt et al., 2016). Mide si el modelo **se equivoca** igual con hombres y mujeres |
| **Error / Accuracy** | $P(\hat Y\neq Y)$ | Objetivo que el paper minimiza; los resultados del paper son curvas error vs. disparidad |
| **Balanced accuracy y AUC** | Promedio de TPR y TNR; área bajo la ROC | Las clases están desbalanceadas (24 % positivos): la accuracy sola es engañosa |
| **Métricas por grupo** | TPR, FPR, FNR, precisión, tasa de selección por sexo | Análisis desagregado exigido en la guía |

Adicionalmente se reporta la **disparidad tal como la define el paper** (máxima desviación de cada grupo
respecto a la media global, que es lo que restringe Fairlearn internamente).

**Tensión esperada:** la tasa base de `>50K` es muy distinta entre sexos (≈11 % mujeres vs. ≈30 % hombres).
Con tasas base diferentes, *Demographic Parity* y *Equalized Odds* no pueden cumplirse a la vez salvo con un
clasificador trivial o perfecto. Por eso entrenamos y comparamos **un modelo mitigado por cada restricción**
en lugar de uno solo.

# 2. Fundamento teórico: Exponentiated Gradient Reduction

**Idea central (reducción).** En lugar de diseñar un algoritmo "justo" nuevo, el paper *reduce* la
clasificación justa a una **secuencia de problemas de clasificación sensible a costos**, que cualquier
clasificador estándar que acepte pesos por muestra puede resolver (regresión logística, árboles, boosting…).

**1. Restricciones como momentos lineales.** Sea $h$ un clasificador. Muchas nociones de equidad de grupo se
escriben como $M\,\mu(h) \le c$, donde $\mu_j(h)=\mathbb{E}[g_j(X,A,Y,h(X))\mid \mathcal{E}_j]$:

- *Demographic Parity:* $\mathbb{E}[h(X)\mid A=a] = \mathbb{E}[h(X)]$ para todo $a$.
- *Equalized Odds:* $\mathbb{E}[h(X)\mid A=a, Y=y] = \mathbb{E}[h(X)\mid Y=y]$ para todo $a, y$.

Cada igualdad se relaja a dos desigualdades con holgura $c$ (en Fairlearn, `difference_bound`).

**2. Clasificadores aleatorizados.** Se busca una distribución $Q$ sobre una familia $\mathcal H$ de
clasificadores: $\min_{Q}\ \text{err}(Q)\ \ \text{s.a.}\ \ M\mu(Q)\le c$. Al permitir mezclas, el problema
se vuelve lineal en $Q$.

**3. Lagrangiano y juego de dos jugadores.**
$$L(Q,\lambda)=\text{err}(Q)+\lambda^\top\big(M\mu(Q)-c\big),\qquad \lambda\ge 0,\ \|\lambda\|_1\le B$$
El problema es el punto de silla $\min_Q \max_\lambda L(Q,\lambda)$, que se resuelve como un juego:

- **Jugador $\lambda$ (el "auditor"):** actualiza los multiplicadores con **exponentiated gradient**
  (pesos multiplicativos), aumentando $\lambda_k$ para las restricciones que más se violan.
- **Jugador $Q$ (el "aprendiz"):** responde con el **mejor clasificador** para los costos actuales. Para
  $\lambda$ fijo, $L$ es un error ponderado, así que basta re-entrenar el clasificador base con
  **etiquetas y pesos de muestra modificados**: un problema de clasificación sensible a costos.

**4. Salida y garantías.** Tras $T$ iteraciones, el promedio de los clasificadores jugados es un punto de
silla $\nu$-aproximado. Con $T = O\!\big(B^2\ln|\mathcal K|/\nu^2\big)$ iteraciones, el resultado es
simultáneamente casi óptimo en error y casi factible en las restricciones (Teorema 1 del paper). La salida es
un **clasificador aleatorizado**: para cada persona se elige al azar uno de los clasificadores $h_t$ según los
pesos $Q(h_t)$.

**Hiperparámetros en Fairlearn**

| Parámetro | Rol en el paper | Valor usado |
|---|---|---|
| `constraints` | Matriz $M$: qué noción de equidad | `DemographicParity`, `EqualizedOdds` |
| `difference_bound` | Holgura $c$ permitida en la restricción | barrido 0.005 – 0.10 |
| `eps` | Precisión $\varepsilon$ del punto de silla; fija $B = 1/\varepsilon$ | 0.01 (default) |
| `max_iter` | Número máximo de iteraciones $T$ | 50 (default) |
| `eta0` | Tasa de aprendizaje inicial del jugador $\lambda$ | 2.0 (default) |

**Supuestos y limitaciones (desde el paper)**

- El atributo protegido $A$ se necesita **solo en entrenamiento**; el clasificador final no lo usa al
  predecir. Esto es una ventaja legal frente a umbrales por grupo (post-procesamiento).
- Las garantías son sobre la **distribución de entrenamiento**: con pocos datos de un grupo, la
  disparidad en test puede ser mayor que la restringida.
- La salida aleatorizada implica que **dos personas idénticas pueden recibir decisiones distintas**: hay que
  justificarlo ante los afectados (o usar el mejor clasificador determinista, perdiendo garantías).
- Solo garantiza equidad **de grupo** para los grupos declarados, no equidad individual.

**Varios atributos protegidos a la vez.** El planteamiento del paper admite cualquier conjunto finito de grupos:
basta con añadir una restricción por grupo. Si se pasan varias columnas como `sensitive_features` (p. ej. sexo y
raza), Fairlearn forma un grupo por cada combinación (*Female/Non-White*, *Male/White*…) y restringe todos. Así se
obtiene equidad **interseccional**, a costa de grupos más pequeños, más restricciones y estimaciones más ruidosas.
Proteger solo los ejes por separado no basta: un modelo puede cumplir paridad por sexo y por raza y aun así
discriminar a las mujeres no blancas (*fairness gerrymandering*, Kearns et al., 2018). Lo comprobamos en la sección 10.

# 3. Carga y análisis exploratorio

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             precision_score, confusion_matrix, roc_curve)

import fairlearn
from fairlearn.datasets import fetch_adult
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import (MetricFrame, count, selection_rate, true_positive_rate,
                               false_positive_rate, false_negative_rate,
                               demographic_parity_difference, demographic_parity_ratio,
                               equalized_odds_difference)

SEED = 42
FIG_DIR = Path("figuras")
FIG_DIR.mkdir(exist_ok=True)

COLORES_GRUPO = {"Female": "#D1495B", "Male": "#00798C"}
COLORES_MODELO = {"Baseline": "#8D99AE", "EG-DP": "#2E86AB", "EG-EO": "#3BB273",
                  "ThrOpt-DP": "#F18F01", "ThrOpt-EO": "#C73E1D"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.png", dpi=160, bbox_inches="tight")

print("fairlearn", fairlearn.__version__)

In [ ]:
adult = fetch_adult(as_frame=True)
X_raw = adult.data.copy()
y = (adult.target == ">50K").astype(int).rename("ingreso_alto")

print("Dimensiones:", X_raw.shape)
print(f"Proporción con ingreso >50K: {y.mean():.2%}")
X_raw.head()

In [ ]:
# Calidad de datos: valores faltantes y duplicados
faltantes = X_raw.isna().sum()
print("Valores faltantes por columna:")
print(faltantes[faltantes > 0])
print("\nFilas duplicadas:", X_raw.assign(y=y).duplicated().sum())

Los faltantes están en `workclass`, `occupation` y `native-country` (en el CSV original aparecen como `?`).
No eliminamos esas filas: el faltante puede no ser aleatorio (p. ej. personas sin empleo formal), y
borrarlas cambiaría la composición de los grupos. Se imputan como una categoría propia, `"Desconocido"`.
Los duplicados son personas distintas con el mismo perfil (el censo no tiene identificador), así que se conservan.

In [ ]:
df = X_raw.assign(ingreso_alto=y)

def resumen_grupo(col):
    r = df.groupby(col, observed=True).agg(n=("ingreso_alto", "size"),
                                           tasa_base=("ingreso_alto", "mean"))
    r["proporcion"] = r.n / r.n.sum()
    return r.sort_values("n", ascending=False).round(3)

display(resumen_grupo("sex"))
display(resumen_grupo("race"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rep = df.sex.value_counts(normalize=True) * 100
axes[0].bar(rep.index, rep.values, color=[COLORES_GRUPO[g] for g in rep.index])
axes[0].set_title("Representación en el dataset (%)")
for i, v in enumerate(rep.values):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center")

tb = df.groupby("sex", observed=True).ingreso_alto.mean() * 100
axes[1].bar(tb.index, tb.values, color=[COLORES_GRUPO[g] for g in tb.index])
axes[1].axhline(y.mean() * 100, color="black", ls="--", lw=1, label="global")
axes[1].set_title("Tasa base de >50K por sexo (%)")
axes[1].legend()
for i, v in enumerate(tb.values):
    axes[1].text(i, v + 0.8, f"{v:.1f}%", ha="center")

horas = df.groupby(["sex", "ingreso_alto"], observed=True)["hours-per-week"].mean().unstack()
horas.plot.bar(ax=axes[2], color=["#BBBBBB", "#555555"], rot=0)
axes[2].set_title("Horas semanales promedio")
axes[2].legend(["<=50K", ">50K"])
plt.tight_layout()
guardar(fig, "01_eda_sexo")
plt.show()

In [ ]:
# ¿Qué variables funcionan como proxy del sexo?
display(pd.crosstab(df.relationship, df.sex, normalize="index").round(3))
display(pd.crosstab(df["marital-status"], df.sex, normalize="index").round(3))

**Hallazgos del EDA**

1. **Sesgo de representación:** las mujeres son ~1/3 de los registros.
2. **Sesgo histórico:** la tasa base de `>50K` en hombres (~30 %) es casi el triple que en mujeres (~11 %).
   Refleja la brecha salarial y la menor participación laboral femenina de EE. UU. en 1994, no una
   diferencia de capacidad. Un modelo entrenado con esta etiqueta aprende y reproduce esa brecha.
3. **Proxies:** `relationship` (`Husband` ≈ 100 % hombres, `Wife` ≈ 100 % mujeres) y `marital-status`
   permiten reconstruir el sexo. Por eso **quitar la columna `sex` no basta** para evitar la discriminación.
4. **Sesgo de medición en la etiqueta:** el ingreso declarado al censo es el ingreso individual. Por eso
   una mujer casada con ingresos del hogar altos aparece como `<=50K`.

## 3.1 Exploración por raza y por sexo × raza

In [ ]:
df["race_bin"] = np.where(df.race == "White", "White", "Non-White")
df["sexo_raza"] = df.sex.astype(str) + " / " + df.race_bin

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
r5 = df.groupby("race", observed=True).ingreso_alto.agg(["mean", "size"]).sort_values("mean")
axes[0].barh([f"{i} (n={n})" for i, n in zip(r5.index, r5["size"])], r5["mean"] * 100, color="#6A4C93")
axes[0].axvline(y.mean() * 100, color="black", ls="--", lw=1)
axes[0].set_title("Tasa base de >50K por raza (%)")
for i, v in enumerate(r5["mean"] * 100):
    axes[0].text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=9)

si = df.groupby("sexo_raza").ingreso_alto.agg(["mean", "size"])
axes[1].bar(si.index, si["mean"] * 100, color=["#D1495B", "#E8909B", "#00798C", "#66AFBA"])
axes[1].axhline(y.mean() * 100, color="black", ls="--", lw=1)
axes[1].set_title("Tasa base de >50K por sexo × raza (%)")
for i, (v, n) in enumerate(zip(si["mean"] * 100, si["size"])):
    axes[1].text(i, v + 0.6, f"{v:.1f}%\nn={n}", ha="center", fontsize=9)
plt.tight_layout(); guardar(fig, "01b_eda_raza"); plt.show()

5. **Sesgo histórico también por raza:** la tasa base de `>50K` es ≈25 % en personas blancas y ≈12 % en personas
   negras. Las personas asiáticas (≈27 %) superan a las blancas, así que agrupar todo en "no blancos" oculta
   situaciones muy distintas.
6. **Interseccionalidad en los datos:** las mujeres no blancas tienen la tasa base más baja de todos los grupos. Su
   desventaja es mayor que la de "ser mujer" o "ser no blanca" por separado. La sección 10 analiza este punto a fondo.

# 4. Preprocesamiento y partición train / validación / test

- **Variables de entrada:** se excluyen `sex` y `race` del modelo (ni el baseline ni el mitigado las usan
  para predecir). También `fnlwgt`, que es un peso muestral del censo y no una característica de la persona.
- **Atributo sensible `A`:** se usa `sex` solo para entrenar EG y para evaluar.
- **Partición 60 / 20 / 20, estratificada por `y` × `sex`:** *train* para ajustar, *validación* para
  elegir la holgura `difference_bound`, *test* solo para el reporte final.
- El preprocesador (one-hot + estandarización) se ajusta **solo con train** para no filtrar información.

In [ ]:
A = X_raw["sex"].astype(str).rename("sex")
RAZA = np.where(X_raw["race"] == "White", "White", "Non-White")
X = X_raw.drop(columns=["sex", "race", "fnlwgt"])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()
X[cat_cols] = X[cat_cols].astype("object").fillna("Desconocido")

estrato = y.astype(str) + "_" + A
idx = np.arange(len(X))
idx_train, idx_tmp = train_test_split(idx, test_size=0.4, stratify=estrato, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=estrato.iloc[idx_tmp], random_state=SEED)

def subset(ix):
    return X.iloc[ix], y.iloc[ix].reset_index(drop=True), A.iloc[ix].reset_index(drop=True), RAZA[ix]

X_train, y_train, A_train, R_train = subset(idx_train)
X_val, y_val, A_val, R_val = subset(idx_val)
X_test, y_test, A_test, R_test = subset(idx_test)

preproc = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols),
])
Z_train = preproc.fit_transform(X_train)
Z_val = preproc.transform(X_val)
Z_test = preproc.transform(X_test)

pd.DataFrame({
    "n": [len(y_train), len(y_val), len(y_test)],
    "% >50K": [y_train.mean(), y_val.mean(), y_test.mean()],
    "% mujeres": [(A_train == "Female").mean(), (A_val == "Female").mean(), (A_test == "Female").mean()],
}, index=["train", "val", "test"]).round(3).assign(features=Z_train.shape[1])

# 5. Modelo *baseline* sin mitigación

Regresión logística (el mismo clasificador base que usa el paper) sin la columna `sex`. Este es el escenario
de *fairness through unawareness*.

Definimos primero las funciones de evaluación que se usarán para **todos** los modelos:

In [ ]:
def disparidad_paper(y_pred, sf, y_true=None):
    """Disparidad como en Agarwal et al.: máxima |E[h|A=a] - E[h]| (DP)
    o máxima |E[h|A=a,Y=y] - E[h|Y=y]| (EO, si se pasa y_true)."""
    y_pred, sf = np.asarray(y_pred), np.asarray(sf)
    if y_true is None:
        return max(abs(y_pred[sf == a].mean() - y_pred.mean()) for a in np.unique(sf))
    y_true = np.asarray(y_true)
    return max(abs(y_pred[(sf == a) & (y_true == v)].mean() - y_pred[y_true == v].mean())
               for a in np.unique(sf) for v in (0, 1))


def evaluar(y_true, y_pred, sf, score=None):
    """Métricas globales de rendimiento y de equidad."""
    kw = dict(y_true=y_true, y_pred=y_pred, sensitive_features=sf)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, score) if score is not None else np.nan,
        "tasa_seleccion": np.mean(y_pred),
        "DPD": demographic_parity_difference(**kw),
        "DPR": demographic_parity_ratio(**kw),
        "EOD": equalized_odds_difference(**kw),
        "disp_DP_paper": disparidad_paper(y_pred, sf),
        "disp_EO_paper": disparidad_paper(y_pred, sf, y_true),
    }


METRICAS_GRUPO = {
    "n": count,
    "tasa_seleccion": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
    "FNR": false_negative_rate,
    "precision": lambda yt, yp: precision_score(yt, yp, zero_division=0),
    "accuracy": accuracy_score,
}

def por_grupo(y_true, y_pred, sf):
    return MetricFrame(metrics=METRICAS_GRUPO, y_true=y_true, y_pred=y_pred, sensitive_features=sf)

In [ ]:
baseline = LogisticRegression(max_iter=2000, random_state=SEED)
baseline.fit(Z_train, y_train)

pred_base_test = baseline.predict(Z_test)
score_base_test = baseline.predict_proba(Z_test)[:, 1]

res_base = evaluar(y_test, pred_base_test, A_test, score_base_test)
pd.Series(res_base).round(4).to_frame("Baseline (test)")

In [ ]:
mf_base = por_grupo(y_test, pred_base_test, A_test)
display(mf_base.by_group.round(3))
print("Diferencias entre grupos (max - min):")
display(mf_base.difference().round(3).to_frame("diferencia").T)

In [ ]:
# Verificación manual con la matriz de confusión (para no depender solo de la librería)
for g in ["Female", "Male"]:
    m = (A_test == g).values
    tn, fp, fn, tp = confusion_matrix(y_test[m], pred_base_test[m], labels=[0, 1]).ravel()
    print(f"{g:7s} TPR={tp/(tp+fn):.3f}  FPR={fp/(fp+tn):.3f}  sel={(tp+fp)/m.sum():.3f}")

In [ ]:
def plot_confusiones(y_true, y_pred, sf, titulo, nombre):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, g in zip(axes, ["Female", "Male"]):
        m = (np.asarray(sf) == g)
        cm = confusion_matrix(np.asarray(y_true)[m], np.asarray(y_pred)[m], labels=[0, 1], normalize="true")
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax,
                    xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"])
        ax.set_title(f"{g} (n={m.sum()})")
        ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    fig.suptitle(titulo + " — matrices de confusión normalizadas por fila")
    plt.tight_layout()
    guardar(fig, nombre)
    plt.show()

plot_confusiones(y_test, pred_base_test, A_test, "Baseline", "02_confusion_baseline")

In [ ]:
def plot_roc_grupos(modelos_scores, y_true, sf, nombre):
    """modelos_scores: dict nombre -> (score, y_pred). Una ROC por grupo y el punto de operación."""
    fig, axes = plt.subplots(1, len(modelos_scores), figsize=(5.2 * len(modelos_scores), 4.4), squeeze=False)
    for ax, (nom, (score, pred)) in zip(axes[0], modelos_scores.items()):
        for g in ["Female", "Male"]:
            m = (np.asarray(sf) == g)
            fpr, tpr, _ = roc_curve(np.asarray(y_true)[m], score[m])
            auc = roc_auc_score(np.asarray(y_true)[m], score[m])
            ax.plot(fpr, tpr, color=COLORES_GRUPO[g], label=f"{g} (AUC={auc:.3f})")
            yt, yp = np.asarray(y_true)[m], np.asarray(pred)[m]
            ax.scatter(yp[yt == 0].mean(), yp[yt == 1].mean(), color=COLORES_GRUPO[g],
                       edgecolor="black", s=70, zorder=3)
        ax.plot([0, 1], [0, 1], color="grey", ls=":", lw=1)
        ax.set_title(nom); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(loc="lower right")
    fig.suptitle("Curvas ROC por sexo (los puntos marcan el punto de operación del clasificador)")
    plt.tight_layout()
    guardar(fig, nombre)
    plt.show()

plot_roc_grupos({"Baseline": (score_base_test, pred_base_test)}, y_test, A_test, "03_roc_baseline")

**Diagnóstico del baseline**

- Accuracy ≈ 0.85, pero el modelo selecciona como `>50K` a una proporción de hombres varias veces mayor que
  de mujeres: el **DPR está muy por debajo de 0.8** (incumple la regla de los 4/5).
- El modelo **no usa `sex`**, y aun así discrimina a través de los proxies (`relationship`, `marital-status`,
  ocupación, horas).
- La EOD es menor que la DPD: buena parte de la diferencia de selección viene de la diferencia de tasas base,
  y el resto de errores desiguales entre grupos (TPR y FPR distintos).
- La AUC por grupo es parecida (incluso mayor en mujeres): el problema no es que el modelo "no sepa" ordenar
  a las mujeres, sino **dónde** cae su punto de operación con un único umbral.

# 6. Mitigación con Exponentiated Gradient: barrido de hiperparámetros

Entrenamos `ExponentiatedGradient` con regresión logística como clasificador base y dos restricciones,
**Demographic Parity (EG-DP)** y **Equalized Odds (EG-EO)**. Para cada una barremos la holgura
`difference_bound` $c$. Como en el paper, esto traza la **frontera error–disparidad**. Todo se mide en
**validación**.

Las predicciones de EG son aleatorizadas, así que se fija `random_state` en `predict`. Para la ROC/AUC se usa
un *score* continuo: el promedio de las probabilidades de los clasificadores base ponderado por la mezcla,
$s(x)=\sum_t Q(h_t)\,P_{h_t}(Y=1\mid x)$. Indica la capacidad de ordenamiento; la decisión real del modelo
sigue siendo la del clasificador aleatorizado.

In [ ]:
COTAS = [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
RESTRICCIONES = {"EG-DP": DemographicParity, "EG-EO": EqualizedOdds}

def entrenar_eg(restriccion, cota, estimador=None, eps=0.01, max_iter=50, sf=None):
    """sf: atributo(s) sensible(s) de entrenamiento; por defecto el sexo. Un DataFrame con varias
    columnas produce grupos interseccionales."""
    est = estimador if estimador is not None else LogisticRegression(max_iter=2000, random_state=SEED)
    eg = ExponentiatedGradient(est, constraints=restriccion(difference_bound=cota),
                               eps=eps, max_iter=max_iter)
    eg.fit(Z_train, y_train, sensitive_features=A_train if sf is None else sf)
    return eg

def score_eg(eg, Z):
    """Score continuo de la mezcla: sum_t Q(h_t) * P_t(Y=1|x) con las probabilidades de cada clasificador base.
    (_pmf_predict promedia salidas 0/1 y da scores casi binarios, con una AUC artificialmente baja.)"""
    return sum(w * h.predict_proba(Z)[:, 1] for w, h in zip(eg.weights_, eg.predictors_) if w > 0)

modelos_eg, filas = {}, []
for nombre, restr in RESTRICCIONES.items():
    for c in COTAS:
        t0 = time.time()
        eg = entrenar_eg(restr, c)
        seg = time.time() - t0
        modelos_eg[(nombre, c)] = eg
        pred = eg.predict(Z_val, random_state=SEED)
        fila = evaluar(y_val, pred, A_val, score_eg(eg, Z_val))
        fila.update(modelo=nombre, cota=c, n_clasificadores=int((eg.weights_ > 0).sum()),
                    iteraciones=len(eg.lambda_vecs_.columns), segundos=round(seg, 1))
        filas.append(fila)
        print(f"{nombre} c={c:<5} acc={fila['accuracy']:.4f} DPD={fila['DPD']:.4f} "
              f"EOD={fila['EOD']:.4f} ({seg:.0f}s)")

base_val = evaluar(y_val, baseline.predict(Z_val), A_val, baseline.predict_proba(Z_val)[:, 1])
barrido = pd.DataFrame(filas)
barrido.to_csv("resultados_barrido_validacion.csv", index=False)
barrido.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metrica, titulo in [(axes[0], "DPD", "Demographic Parity Difference"),
                            (axes[1], "EOD", "Equalized Odds Difference")]:
    for nombre in RESTRICCIONES:
        sub = barrido[barrido.modelo == nombre].sort_values(metrica)
        ax.plot(sub[metrica], sub.accuracy, "o-", color=COLORES_MODELO[nombre], label=nombre)
        for _, r in sub.iterrows():
            ax.annotate(f"c={r.cota}", (r[metrica], r.accuracy), fontsize=7,
                        xytext=(4, 4), textcoords="offset points", color=COLORES_MODELO[nombre])
    ax.scatter(base_val[metrica], base_val["accuracy"], marker="*", s=250,
               color=COLORES_MODELO["Baseline"], edgecolor="black", label="Baseline", zorder=3)
    ax.set_xlabel(titulo + " (↓ más justo)"); ax.set_ylabel("Accuracy (validación)")
    ax.set_title(f"Trade-off accuracy vs. {metrica}")
    ax.legend()
plt.tight_layout()
guardar(fig, "04_tradeoff_validacion")
plt.show()

**Lectura de las curvas de trade-off**

- **EG-DP** traza una frontera clara: al reducir la cota $c$, la DPD cae de ~0.17 a casi 0 con una pérdida de
  accuracy de 1 a 2 puntos. El resultado es consistente con la Figura 1 del paper para Adult.
- **EG-EO** baja la EOD a ~0.02 con un costo de accuracy aún menor. Con cotas grandes ($c \ge 0.05$) la
  restricción casi no se activa y el modelo queda muy cerca del baseline.
- **Tensión entre métricas:** EG-DP *empeora* la EOD (para igualar tasas de selección con tasas base
  distintas debe aceptar más falsos positivos en mujeres o más falsos negativos en hombres). EG-EO apenas mueve
  la DPD. Es la incompatibilidad esperada entre ambas nociones.

**Criterio de selección (en validación).** Para cada restricción elegimos la mayor accuracy entre los modelos
cuya métrica objetivo sea ≤ **0.03**. Es una tolerancia pequeña en puntos porcentuales y, para DP, garantiza
holgadamente la regla de los 4/5.

In [ ]:
TOLERANCIA = 0.03
seleccion = {}
for nombre, metrica in [("EG-DP", "DPD"), ("EG-EO", "EOD")]:
    cand = barrido[(barrido.modelo == nombre) & (barrido[metrica] <= TOLERANCIA)]
    if cand.empty:  # si ninguna cumple, la más justa
        cand = barrido[barrido.modelo == nombre].nsmallest(1, metrica)
    mejor = cand.sort_values("accuracy", ascending=False).iloc[0]
    seleccion[nombre] = mejor.cota
    print(f"{nombre}: cota elegida c={mejor.cota}  (val: acc={mejor.accuracy:.4f}, {metrica}={mejor[metrica]:.4f})")

eg_dp = modelos_eg[("EG-DP", seleccion["EG-DP"])]
eg_eo = modelos_eg[("EG-EO", seleccion["EG-EO"])]

In [ ]:
# ¿Cómo es por dentro el clasificador aleatorizado elegido?
for nom, eg in [("EG-DP", eg_dp), ("EG-EO", eg_eo)]:
    w = eg.weights_[eg.weights_ > 0].sort_values(ascending=False)
    print(f"{nom}: {len(w)} clasificadores con peso > 0 de {len(eg.predictors_)} generados; "
          f"pesos principales = {np.round(w.values[:5], 3)}")

fig, ax = plt.subplots(figsize=(8, 3.5))
lam = eg_dp.lambda_vecs_
for col in lam.index:
    ax.plot(lam.loc[col].values, label=str(col))
ax.set_xlabel("Iteración"); ax.set_ylabel("λ")
ax.set_title("EG-DP: evolución de los multiplicadores de Lagrange (jugador λ)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); guardar(fig, "05_lambdas"); plt.show()

La gráfica muestra el juego del paper: los multiplicadores $\lambda$ de las restricciones que se violan (la
tasa de selección de mujeres por debajo de la media y la de hombres por encima) crecen en las primeras
iteraciones y después se estabilizan cuando el aprendiz encuentra una mezcla factible. El resultado final es
una **mezcla de pocas regresiones logísticas** entrenadas con pesos distintos.

# 7. Evaluación final en test: antes vs. después

In [ ]:
pred_test = {
    "Baseline": pred_base_test,
    "EG-DP": eg_dp.predict(Z_test, random_state=SEED),
    "EG-EO": eg_eo.predict(Z_test, random_state=SEED),
}
score_test = {
    "Baseline": score_base_test,
    "EG-DP": score_eg(eg_dp, Z_test),
    "EG-EO": score_eg(eg_eo, Z_test),
}

tabla_final = pd.DataFrame({m: evaluar(y_test, pred_test[m], A_test, score_test[m]) for m in pred_test}).T
tabla_final.to_csv("resultados_test.csv")
tabla_final.round(4)

In [ ]:
# Cambio relativo respecto al baseline
cambio = (tabla_final - tabla_final.loc["Baseline"]).drop("Baseline")
cambio[["accuracy", "balanced_acc", "AUC", "DPD", "EOD"]].round(4)

In [ ]:
# Métricas desagregadas por sexo, para los tres modelos
desag = pd.concat({m: por_grupo(y_test, pred_test[m], A_test).by_group for m in pred_test})
desag.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, met in zip(axes, ["tasa_seleccion", "TPR", "FPR", "precision"]):
    piv = desag[met].unstack()
    piv.plot.bar(ax=ax, color=[COLORES_GRUPO[c] for c in piv.columns], rot=0, width=0.75)
    ax.set_title(met); ax.set_xlabel(""); ax.legend(title="")
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.2f", fontsize=7)
fig.suptitle("Métricas por sexo en test: baseline vs. modelos mitigados")
plt.tight_layout(); guardar(fig, "06_barras_por_grupo"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
mets = ["accuracy", "balanced_acc", "DPD", "EOD", "DPR"]
x = np.arange(len(mets)); ancho = 0.26
for i, m in enumerate(pred_test):
    vals = tabla_final.loc[m, mets].values.astype(float)
    b = ax.bar(x + (i - 1) * ancho, vals, ancho, color=COLORES_MODELO[m], label=m)
    ax.bar_label(b, fmt="%.3f", fontsize=7)
ax.axhline(0.8, color="black", ls=":", lw=1)
ax.text(len(mets) - 0.5, 0.82, "regla 4/5 (DPR)", fontsize=8, ha="right")
ax.set_xticks(x); ax.set_xticklabels(mets)
ax.set_title("Comparación global en test")
ax.legend()
plt.tight_layout(); guardar(fig, "07_comparacion_global"); plt.show()

In [ ]:
plot_confusiones(y_test, pred_test["EG-DP"], A_test, "EG-DP", "08_confusion_eg_dp")
plot_confusiones(y_test, pred_test["EG-EO"], A_test, "EG-EO", "09_confusion_eg_eo")

In [ ]:
plot_roc_grupos({m: (score_test[m], pred_test[m]) for m in pred_test}, y_test, A_test, "10_roc_comparacion")

**Interpretación**

- **EG-DP** reduce la brecha de tasa de selección de 0.18 a ~0.03 (DPR de 0.30 a > 0.8: cumple la regla de
  los 4/5), con ~1 punto menos de accuracy (~4 de balanced accuracy). En test la DPD queda algo por encima de
  la cota elegida (0.02): la garantía es sobre la muestra de entrenamiento. Lo consigue **subiendo la TPR y la FPR de las mujeres y bajando las de los hombres**:
  la EOD aumenta. En términos de la ROC, desplaza el punto de operación de cada grupo sobre su propia curva.
- **EG-EO** acerca TPR y FPR entre sexos (EOD de 0.085 a ~0.05) con menos de 1 punto de pérdida de accuracy,
  pero mantiene una diferencia importante en tasa de selección, porque las tasas base reales siguen siendo distintas.
- La **AUC del score de la mezcla** cambia poco: la mitigación conserva la capacidad de ordenar del modelo y
  sobre todo reubica los puntos de decisión de cada grupo.
- **Precisión por grupo:** con EG-DP la precisión en mujeres baja de 0.74 a ~0.57. Más mujeres reciben la
  decisión favorable, pero una fracción mayor de ellas son falsos positivos: es el costo de imponer paridad
  demográfica con tasas base distintas.
- **¿Qué modelo elegir?** Depende de la noción de justicia del contexto (sección 10 e informe).

# 8. Análisis de sensibilidad y robustez

## 8.1 Variabilidad por la aleatorización de las predicciones

In [ ]:
semillas = range(30)
var = []
for m, eg in [("EG-DP", eg_dp), ("EG-EO", eg_eo)]:
    for s in semillas:
        p = eg.predict(Z_test, random_state=s)
        var.append({"modelo": m, "semilla": s, **evaluar(y_test, p, A_test)})
var = pd.DataFrame(var)
resumen_var = var.groupby("modelo")[["accuracy", "DPD", "EOD"]].agg(["mean", "std", "min", "max"]).round(4)
display(resumen_var)

# ¿Cuántas personas cambian de decisión según la semilla?
P = np.array([eg_dp.predict(Z_test, random_state=s) for s in semillas])
inestables = (P.min(axis=0) != P.max(axis=0)).mean()
print(f"EG-DP: {inestables:.1%} de las personas de test reciben decisiones distintas según la semilla")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.boxplot(data=var, x="modelo", y="accuracy", ax=axes[0], palette=COLORES_MODELO)
sns.boxplot(data=var.melt(id_vars="modelo", value_vars=["DPD", "EOD"]), x="variable", y="value",
            hue="modelo", ax=axes[1], palette=COLORES_MODELO)
axes[0].set_title("Accuracy en 30 semillas de predicción")
axes[1].set_title("Disparidad en 30 semillas de predicción")
plt.tight_layout(); guardar(fig, "11_variabilidad_semillas"); plt.show()

La variación entre semillas es pequeña en las métricas agregadas (desviación estándar del orden de 10⁻³),
pero **a nivel individual** una fracción no despreciable de personas puede recibir una decisión distinta por
puro azar. Las garantías del paper son sobre esperanzas, no sobre cada individuo. Esto es un argumento
importante para el análisis ético y para la supervisión humana (Art. 14 del AI Act).

## 8.2 Sensibilidad a `eps` y `max_iter` (EG-DP con la cota elegida)

In [ ]:
filas = []
for eps in [0.002, 0.01, 0.1]:
    for it in [3, 50]:
        t0 = time.time()
        eg = entrenar_eg(DemographicParity, seleccion["EG-DP"], eps=eps, max_iter=it)
        p = eg.predict(Z_val, random_state=SEED)
        filas.append({"eps": eps, "max_iter": it, "segundos": round(time.time() - t0, 1),
                      "n_clasificadores": int((eg.weights_ > 0).sum()),
                      **{k: v for k, v in evaluar(y_val, p, A_val).items() if k in ["accuracy", "DPD", "EOD"]}})
sens_hiper = pd.DataFrame(filas)
sens_hiper.round(4)

`eps` controla la precisión del punto de silla ($B = 1/arepsilon$) y `max_iter` el número máximo de
rondas del juego. Los resultados son **estables frente a `eps`** en todo el rango probado. Con `max_iter=3`
la disparidad ya cumple la cota (el jugador λ corrige rápido), pero la **accuracy cae ~2 puntos**: el juego no
alcanza a encontrar la mezcla de menor error dentro de la región factible. Con el valor por defecto (50) el
algoritmo converge. El parámetro que realmente controla el trade-off es la cota `difference_bound` (sección 6).

## 8.3 Cambio de clasificador base: árbol de decisión

In [ ]:
arbol = DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=SEED).fit(Z_train, y_train)
eg_arbol = entrenar_eg(DemographicParity, seleccion["EG-DP"],
                       estimador=DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=SEED))
comp_base = pd.DataFrame({
    "Árbol baseline": evaluar(y_test, arbol.predict(Z_test), A_test, arbol.predict_proba(Z_test)[:, 1]),
    "Árbol + EG-DP": evaluar(y_test, eg_arbol.predict(Z_test, random_state=SEED), A_test, score_eg(eg_arbol, Z_test)),
    "LogReg baseline": tabla_final.loc["Baseline"],
    "LogReg + EG-DP": tabla_final.loc["EG-DP"],
}).T
comp_base[["accuracy", "balanced_acc", "AUC", "DPD", "EOD"]].round(4)

La reducción es **agnóstica al modelo**, como dice el paper: con un árbol de decisión EG también reduce la
DPD a niveles similares con una pérdida de accuracy comparable. Esto confirma que la técnica no depende de la
regresión logística.

# 9. Comparación con una alternativa de post-procesamiento

Como referencia usamos `ThresholdOptimizer` (Hardt et al., 2016), el método de post-procesamiento con el que
el paper compara EG. Aplica **umbrales distintos por grupo** sobre el baseline ya entrenado. Necesita el
sexo **al momento de predecir**, lo que tiene implicaciones legales (tratamiento dispar explícito).

In [ ]:
for nombre, restr in [("ThrOpt-DP", "demographic_parity"), ("ThrOpt-EO", "equalized_odds")]:
    to = ThresholdOptimizer(estimator=baseline, constraints=restr, prefit=True,
                            predict_method="predict_proba", objective="accuracy_score")
    to.fit(Z_train, y_train, sensitive_features=A_train)
    pred_test[nombre] = to.predict(Z_test, sensitive_features=A_test, random_state=SEED)

tabla_comp = pd.DataFrame({m: evaluar(y_test, pred_test[m], A_test) for m in pred_test}).T
tabla_comp["usa_sexo_al_predecir"] = ["No", "No", "No", "Sí", "Sí"]
tabla_comp[["accuracy", "balanced_acc", "DPD", "DPR", "EOD", "usa_sexo_al_predecir"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, met in zip(axes, ["DPD", "EOD"]):
    for m, r in tabla_comp.iterrows():
        ax.scatter(r[met], r.accuracy, s=140, color=COLORES_MODELO[m], edgecolor="black", label=m, zorder=3)
    for nombre in RESTRICCIONES:
        sub = barrido[barrido.modelo == nombre].sort_values(met)
        ax.plot(sub[met], sub.accuracy, ":", color=COLORES_MODELO[nombre], alpha=.6)
    ax.set_xlabel(met + " (test, ↓ más justo)"); ax.set_ylabel("Accuracy (test)")
    ax.set_title(f"In-processing (EG) vs. post-processing — {met}")
axes[0].legend(fontsize=8)
plt.tight_layout(); guardar(fig, "13_eg_vs_postproc"); plt.show()

El post-procesamiento alcanza disparidades algo menores (puede usar un umbral distinto para cada sexo), con
una accuracy ligeramente inferior a EG. Esto coincide con el paper, donde ambos métodos quedan cerca en Adult. La diferencia está en **cómo** lo logran:
`ThresholdOptimizer` necesita conocer el sexo de cada persona al decidir, y bajo ECOA o el principio de no
discriminación de la UE eso puede constituir *trato dispar*. EG produce un modelo que **no recibe el sexo
como entrada** en producción; solo lo usó durante el entrenamiento.

# 10. Múltiples formas de discriminación: raza, interseccionalidad y edad

Hasta aquí protegimos un solo atributo (sexo). Pero la discriminación no ocurre en un solo eje. Esta sección
amplía el análisis a la **raza**, a la **intersección sexo × raza** y a la **edad**, y conecta cada hallazgo con
una forma de discriminación reconocida en la literatura y en el derecho.

## 10.1 Tipología: ¿de qué formas puede discriminar un modelo?

| Forma de discriminación | Definición | Dónde aparece en este proyecto |
|---|---|---|
| **Directa** (*disparate treatment*) | La decisión usa explícitamente el atributo protegido | Nuestros modelos **no** reciben sexo ni raza. `ThresholdOptimizer` sí (umbral por grupo). La **edad** sí es una entrada del modelo |
| **Indirecta** (*disparate impact*) | Una regla aparentemente neutral produce resultados desiguales entre grupos | DPR por sexo = 0.30 y por raza ≈ 0.56 en el baseline |
| **Por proxy** | Variables correlacionadas reconstruyen el atributo protegido | `relationship` → sexo; `native-country` → raza (el 72 % de las personas asiáticas no nació en EE. UU.) |
| **Histórica / estructural** | La etiqueta registra desigualdades pasadas | Tasas base de `>50K`: 30 % hombres vs 11 % mujeres; 25 % blancos vs 12 % negros |
| **Interseccional** (Crenshaw, 1989) | La desventaja de una combinación de atributos es mayor (o distinta) que la suma de cada eje | Mujeres no blancas: ≈5 % de selección frente a ≈27 % de hombres blancos |
| **Por agregación / invisibilidad** | Agrupar categorías o tener grupos pequeños oculta disparidades | "Non-White" mezcla personas negras (≈9 %) con asiáticas (≈24 %); solo 97 personas Amer-Indian en test |
| **Estadística** | Atribuir a la persona el promedio de su grupo | Edad: el modelo penaliza a los jóvenes porque en promedio ganan menos |

Las tres primeras son las que **regulan** las leyes antidiscriminación (trato vs. impacto). La interseccional y la de
agregación son las más difíciles de detectar, porque **no aparecen si se audita un eje a la vez**.

## 10.2 Auditoría racial del baseline

In [ ]:
RAZA5_test = X_raw["race"].astype(str).values[idx_test]
INTER_test = A_test.values + " / " + R_test

def bootstrap_seleccion(pred, grupos, n_boot=500, seed=SEED):
    """Tasa de selección por grupo con intervalo de confianza bootstrap del 95 %."""
    rng = np.random.default_rng(seed)
    pred, grupos = np.asarray(pred), np.asarray(grupos)
    cats = sorted(np.unique(grupos))
    boots = np.empty((n_boot, len(cats)))
    for b in range(n_boot):
        ix = rng.integers(0, len(pred), len(pred))
        pb, gb = pred[ix], grupos[ix]
        boots[b] = [pb[gb == c].mean() if (gb == c).any() else np.nan for c in cats]
    return pd.DataFrame({"n": [(grupos == c).sum() for c in cats],
                         "tasa_base": [np.asarray(y_test)[grupos == c].mean() for c in cats],
                         "seleccion": [pred[grupos == c].mean() for c in cats],
                         "IC95_inf": np.nanpercentile(boots, 2.5, axis=0),
                         "IC95_sup": np.nanpercentile(boots, 97.5, axis=0)}, index=cats)

aud_raza5 = bootstrap_seleccion(pred_base_test, RAZA5_test)
display(aud_raza5.round(3))

mf_raza5 = por_grupo(y_test, pred_base_test, RAZA5_test)
display(mf_raza5.by_group[["n", "tasa_seleccion", "TPR", "FPR", "precision"]].round(3))

print("Baseline, raza binaria (White vs Non-White):")
pd.Series(evaluar(y_test, pred_base_test, R_test)).round(4).to_frame("raza").T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
orden = aud_raza5.sort_values("seleccion").index
d = aud_raza5.loc[orden]
axes[0].barh(orden, d.tasa_base, color="#CCCCCC", label="tasa base real")
axes[0].errorbar(d.seleccion, orden, xerr=[d.seleccion - d.IC95_inf, d.IC95_sup - d.seleccion],
                 fmt="o", color="#2B2D42", capsize=4, label="selección del modelo (IC 95 %)")
for i, (g, r) in enumerate(d.iterrows()):
    axes[0].text(0.005, i, f"n={r.n}", va="center", fontsize=8, color="white" if r.tasa_base > 0.05 else "black")
axes[0].axvline(0.8 * d.seleccion.max(), color="#D1495B", ls=":", lw=1.2, label="4/5 del grupo más seleccionado")
axes[0].set_title("Baseline: selección por raza (5 categorías)")
axes[0].legend(fontsize=8, loc="lower right")

inter_base = bootstrap_seleccion(pred_base_test, INTER_test)
colores_inter = ["#D1495B", "#E8909B", "#00798C", "#66AFBA"]
axes[1].bar(inter_base.index, inter_base.seleccion, color=colores_inter,
            yerr=[inter_base.seleccion - inter_base.IC95_inf, inter_base.IC95_sup - inter_base.seleccion], capsize=4)
for i, v in enumerate(inter_base.seleccion):
    axes[1].text(i, v + 0.012, f"{v:.3f}", ha="center")
axes[1].set_title("Baseline: selección por sexo × raza")
axes[1].set_ylabel("P(Ŷ = 1)")
plt.tight_layout(); guardar(fig, "14_auditoria_raza"); plt.show()

**Hallazgos de la auditoría racial (baseline)**

- **Impacto dispar por raza:** con la variable binaria, la tasa de selección de personas no blancas es ≈56 % de la
  de personas blancas. **Incumple la regla de los 4/5**, igual que el sexo, aunque la brecha es menor.
- **La categoría "Non-White" oculta diferencias enormes:** las personas negras tienen una selección de ≈9 %, pero las
  asiáticas ≈24 %, por encima de las blancas. Agregar en "blanco / no blanco" promedia grupos con situaciones
  opuestas (**discriminación por agregación**).
- **Grupos invisibles:** Amer-Indian-Eskimo (n≈97) y Other (n≈79) tienen tasas de selección muy bajas, pero sus
  intervalos de confianza son anchos: con tan pocos datos, ninguna métrica es fiable y ninguna garantía estadística
  los protege.
- **Interseccionalidad:** las **mujeres no blancas** son seleccionadas ≈5 % de las veces frente a ≈27 % de los
  hombres blancos (razón ≈0.19). La brecha conjunta es **mayor que la de cualquiera de los dos ejes por separado**.

## 10.3 Proxies de raza

In [ ]:
proxy_raza = pd.crosstab(X_raw["race"], X_raw["native-country"].astype(object).fillna("Desconocido") == "United-States",
                         normalize="index").round(3)
proxy_raza.columns = ["Nacido fuera de EE. UU.", "Nacido en EE. UU."]
display(proxy_raza)
display(pd.crosstab(X_raw["race"], X_raw["relationship"], normalize="index").round(3)[["Husband", "Wife", "Unmarried", "Own-child"]])

`native-country` funciona como proxy de raza para las personas asiáticas (72 % nacidas fuera de EE. UU.) y "Other".
`relationship` también varía por raza: la proporción de `Unmarried` es mucho mayor en personas negras, y esa categoría
está asociada a ingresos bajos. Por eso, igual que con el sexo, **excluir `race` no impide la discriminación racial**.

## 10.4 Mitigación por raza y efectos cruzados entre ejes

Entrenamos Exponentiated Gradient con **raza** como atributo sensible (White vs Non-White), con la misma cota
$c = 0.02$ elegida para el sexo, y medimos todos los modelos en **los tres ejes**: sexo, raza y sexo × raza.
Usamos la raza binaria para la restricción porque las categorías pequeñas (n < 300 en train para dos de ellas) harían
la restricción estadísticamente inestable. La desagregación en 5 categorías se usa para **auditar**.

In [ ]:
EJES_TEST = {"sexo": A_test.values, "raza": R_test, "sexo×raza": INTER_test}
EJES_VAL = {"sexo": A_val.values, "raza": R_val, "sexo×raza": A_val.values + " / " + R_val}

def evaluar_ejes(pred, y_true, ejes):
    fila = {"accuracy": accuracy_score(y_true, pred), "balanced_acc": balanced_accuracy_score(y_true, pred)}
    for eje, sf in ejes.items():
        kw = dict(y_true=y_true, y_pred=pred, sensitive_features=sf)
        fila[f"DPD {eje}"] = demographic_parity_difference(**kw)
        fila[f"DPR {eje}"] = demographic_parity_ratio(**kw)
        fila[f"EOD {eje}"] = equalized_odds_difference(**kw)
    return fila

SF_RAZA_TRAIN = pd.Series(R_train, name="race")
SF_INTER_TRAIN = pd.DataFrame({"sex": A_train.values, "race": R_train})

modelos_ejes = {"Baseline": baseline, "EG-DP sexo": eg_dp, "EG-EO sexo": eg_eo}
for nombre, restr in [("EG-DP raza", DemographicParity), ("EG-EO raza", EqualizedOdds)]:
    t0 = time.time()
    modelos_ejes[nombre] = entrenar_eg(restr, 0.02, sf=SF_RAZA_TRAIN)
    print(f"{nombre}: {time.time() - t0:.0f}s")

def predecir(m, Z):
    return m.predict(Z) if m is baseline else m.predict(Z, random_state=SEED)

cruce = pd.DataFrame({n: evaluar_ejes(predecir(m, Z_test), y_test, EJES_TEST) for n, m in modelos_ejes.items()}).T
cruce[["accuracy", "DPR sexo", "DPR raza", "DPR sexo×raza", "EOD sexo", "EOD raza"]].round(3)

**Mitigar un eje no corrige el otro.**

- **EG-DP por raza** lleva la DPR racial por encima de 0.8, pero la DPR por sexo apenas se mueve: **las mujeres
  siguen discriminadas**.
- **EG-DP por sexo** corrige el sexo, pero la DPR racial sigue por debajo de 0.8.
- Cada modelo es "justo" en el eje que se le pidió y sigue discriminando en el otro. Kearns et al. (2018) llaman a
  esto **fairness gerrymandering**: cumplir la restricción en grupos marginales sin protegerlos en sus intersecciones.

## 10.5 Mitigación interseccional (sexo × raza)

Fairlearn permite pasar **varias columnas** como `sensitive_features`: crea un grupo por cada combinación
(4 grupos: Female/Non-White, Female/White, Male/Non-White, Male/White) e impone la restricción sobre todos ellos.
Para elegir la cota hacemos un barrido en validación con el mismo criterio de antes (máxima accuracy con disparidad
interseccional ≤ 0.03; si ninguna cumple, la de menor disparidad).

In [ ]:
COTAS_INTER = [0.01, 0.02, 0.03, 0.05]
filas, eg_inter = [], {}
for nombre, restr, metrica in [("EG-DP sexo×raza", DemographicParity, "DPD sexo×raza"),
                               ("EG-EO sexo×raza", EqualizedOdds, "EOD sexo×raza")]:
    for c in COTAS_INTER:
        t0 = time.time()
        eg = entrenar_eg(restr, c, sf=SF_INTER_TRAIN)
        eg_inter[(nombre, c)] = eg
        fila = evaluar_ejes(eg.predict(Z_val, random_state=SEED), y_val, EJES_VAL)
        fila.update(modelo=nombre, cota=c)
        filas.append(fila)
        print(f"{nombre} c={c:<5} acc={fila['accuracy']:.4f} {metrica}={fila[metrica]:.4f} ({time.time() - t0:.0f}s)")
barrido_inter = pd.DataFrame(filas)

for nombre, metrica in [("EG-DP sexo×raza", "DPD sexo×raza"), ("EG-EO sexo×raza", "EOD sexo×raza")]:
    sub = barrido_inter[barrido_inter.modelo == nombre]
    cand = sub[sub[metrica] <= TOLERANCIA]
    elegido = (cand if not cand.empty else sub.nsmallest(1, metrica)).sort_values("accuracy").iloc[-1]
    modelos_ejes[nombre] = eg_inter[(nombre, elegido.cota)]
    print(f"→ {nombre}: cota elegida c={elegido.cota}")

barrido_inter.to_csv("resultados_barrido_interseccional.csv", index=False)
barrido_inter[["modelo", "cota", "accuracy", "DPR sexo", "DPR raza", "DPR sexo×raza", "EOD sexo×raza"]].round(4)

In [ ]:
pred_ejes = {n: predecir(m, Z_test) for n, m in modelos_ejes.items()}
cruce = pd.DataFrame({n: evaluar_ejes(p, y_test, EJES_TEST) for n, p in pred_ejes.items()}).T
cruce.to_csv("resultados_multi_eje_test.csv")
cruce.round(3)

## 10.6 Matriz de disparidades: cada modelo evaluado en cada eje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1, 1]})
etiquetas = [f"{n}  (acc {cruce.loc[n, 'accuracy']:.3f})" for n in cruce.index]
dpr = cruce[["DPR sexo", "DPR raza", "DPR sexo×raza"]]
sns.heatmap(dpr, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, center=0.8, ax=axes[0],
            yticklabels=etiquetas, cbar_kws={"label": "DPR (≥ 0.8 cumple 4/5)"})
axes[0].set_title("Razón de paridad demográfica (↑ mejor)")
eodm = cruce[["EOD sexo", "EOD raza", "EOD sexo×raza"]]
sns.heatmap(eodm, annot=True, fmt=".3f", cmap="RdYlGn_r", vmin=0, vmax=0.35, ax=axes[1],
            yticklabels=False, cbar_kws={"label": "EOD"})
axes[1].set_title("Diferencia de equalized odds (↓ mejor)")
for ax in axes:
    ax.set_xticklabels(["sexo", "raza", "sexo × raza"], rotation=0)
plt.tight_layout(); guardar(fig, "15_matriz_ejes"); plt.show()

In [ ]:
sel_inter = pd.DataFrame({m: pd.Series(pred_ejes[m]).groupby(INTER_test).mean()
                          for m in ["Baseline", "EG-DP sexo", "EG-DP raza", "EG-DP sexo×raza"]})
display(sel_inter.round(3).rename_axis("tasa de selección por sexo × raza"))

fig, ax = plt.subplots(figsize=(11, 4.2))
colores = {"Baseline": "#8D99AE", "EG-DP sexo": "#2E86AB", "EG-DP raza": "#F18F01", "EG-DP sexo×raza": "#6A4C93"}
sel_inter.plot.bar(ax=ax, color=[colores[c] for c in sel_inter.columns], rot=0, width=0.8)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=7)
ax.set_title("Tasa de selección por grupo interseccional según el eje que se mitiga")
ax.set_ylabel("P(Ŷ = 1)"); ax.set_xlabel("")
ax.legend(title="Modelo")
plt.tight_layout(); guardar(fig, "16_interseccional_modelos"); plt.show()

**Lectura de la matriz y del gráfico**

- **Solo el modelo interseccional cumple la regla de los 4/5 en los tres ejes a la vez.** EG-DP sexo×raza iguala la
  tasa de selección de los cuatro grupos, con una pérdida de accuracy algo mayor que los modelos de un solo eje: es el
  precio de proteger más grupos.
- **El modelo que solo mitiga sexo "le cobra" a los hombres no blancos:** su selección baja por debajo de la de las
  mujeres blancas. **El que solo mitiga raza deja intacta la brecha de género.** Ambos son ejemplos de
  *fairness gerrymandering*.
- **Equalized odds interseccional es más difícil:** al dividir en 4 grupos, el grupo de mujeres no blancas tiene muy
  pocos positivos reales (ver 10.7), así que su TPR se estima con mucho ruido y la EOD interseccional en test no
  baja como en validación. Es una limitación estadística, no del algoritmo.

## 10.7 ¿Son fiables estas métricas? Tamaño de grupo e intervalos de confianza

In [ ]:
tam = pd.DataFrame({"n": pd.Series(INTER_test).value_counts(),
                    "positivos reales (Y=1)": pd.Series(INTER_test[np.asarray(y_test) == 1]).value_counts()})
display(tam)

def bootstrap_metrica(pred, sf, fn, n_boot=300, seed=SEED):
    rng = np.random.default_rng(seed)
    yt, pred, sf = np.asarray(y_test), np.asarray(pred), np.asarray(sf)
    vals = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(yt), len(yt))
        vals.append(fn(yt[ix], pred[ix], sensitive_features=sf[ix]))
    return np.percentile(vals, [2.5, 97.5])

ic = []
for m in ["Baseline", "EG-DP sexo", "EG-DP sexo×raza", "EG-EO sexo×raza"]:
    for eje in ["sexo", "sexo×raza"]:
        for nom, fn in [("DPR", demographic_parity_ratio), ("EOD", equalized_odds_difference)]:
            lo, hi = bootstrap_metrica(pred_ejes[m], EJES_TEST[eje], fn)
            ic.append({"modelo": m, "eje": eje, "métrica": nom, "valor": cruce.loc[m, f"{nom} {eje}"],
                       "IC95_inf": lo, "IC95_sup": hi})
ic = pd.DataFrame(ic)
ic.round(3)

Con ~9.800 personas en test, las métricas por sexo tienen intervalos estrechos. Las interseccionales, sobre todo la
**EOD**, tienen intervalos mucho más anchos, porque el grupo de mujeres no blancas tiene pocas decenas de positivos.
**Conclusión metodológica:** cuantos más grupos se protegen, más datos se necesitan por grupo. Con grupos pequeños,
la auditoría debe reportar intervalos de confianza, y la decisión debe complementarse con revisión cualitativa.

## 10.8 Edad: auditoría de discriminación por edad

La guía cita la **Age Discrimination in Employment Act (ADEA)**, que en EE. UU. protege a las personas de **40 años o
más** en el empleo. En la UE, la Directiva 2000/78/CE prohíbe la discriminación por edad **a cualquier edad** en el
empleo. A diferencia de sexo y raza, **la edad sí es una variable de entrada** de nuestros modelos.

In [ ]:
EDAD_test = X_raw["age"].values[idx_test]
TRAMO_test = pd.cut(EDAD_test, [16, 24, 39, 54, 100], labels=["17-24", "25-39", "40-54", "55+"]).astype(str)
aud_edad = pd.DataFrame({
    "n": pd.Series(TRAMO_test).value_counts().sort_index(),
    "tasa_base": pd.Series(np.asarray(y_test)).groupby(TRAMO_test).mean(),
    **{f"sel {m}": pd.Series(pred_ejes[m]).groupby(TRAMO_test).mean() for m in ["Baseline", "EG-DP sexo×raza"]},
})
display(aud_edad.round(3))

ADEA_test = np.where(EDAD_test >= 40, "40+", "<40")
print("Razón de selección <40 / 40+ (ADEA):",
      {m: round(demographic_parity_ratio(y_test, pred_ejes[m], sensitive_features=ADEA_test), 3)
       for m in ["Baseline", "EG-DP sexo×raza"]})

fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(aud_edad))
ax.bar(x - 0.27, aud_edad.tasa_base, 0.27, color="#CCCCCC", label="tasa base real")
ax.bar(x, aud_edad["sel Baseline"], 0.27, color="#8D99AE", label="Baseline")
ax.bar(x + 0.27, aud_edad["sel EG-DP sexo×raza"], 0.27, color="#6A4C93", label="EG-DP sexo×raza")
ax.set_xticks(x); ax.set_xticklabels(aud_edad.index)
ax.set_xlabel("Tramo de edad"); ax.set_ylabel("P(Ŷ = 1)")
ax.set_title("Selección por tramo de edad")
ax.legend()
plt.tight_layout(); guardar(fig, "17_edad"); plt.show()

**Lectura:** el grupo protegido por la ADEA (40+) **no** sale perjudicado: tiene la mayor tasa de selección. Los
perjudicados son los **jóvenes de 17 a 24 años**, que casi nunca son seleccionados. Esto plantea una pregunta
distinta a la de sexo y raza:

- La edad tiene una relación **causal plausible** con el ingreso (experiencia, antigüedad). Parte de la disparidad
  puede estar **justificada** por el propósito del sistema (estimar capacidad de pago).
- Pero usar la edad como entrada es **trato directo**. Bajo el derecho europeo solo es lícito si persigue una
  finalidad legítima y es proporcionado. Además, rechazar sistemáticamente a los jóvenes les impide construir el
  historial que el propio sistema exige (**bucle de retroalimentación**).
- **Decisión:** no mitigamos la edad con EG, porque imponer paridad sobre una variable legítimamente predictiva
  sería difícil de justificar. Sí la **documentamos y auditamos**, como pide el Art. 10 del AI Act. La justificación de
  usarla debe constar en la documentación técnica.

## 10.9 Síntesis de la sección

| Pregunta | Respuesta |
|---|---|
| ¿Discrimina el baseline por raza? | Sí: DPR ≈ 0.56 (incumple 4/5). Las personas negras y las categorías pequeñas son las más afectadas |
| ¿Mitigar el sexo resuelve la raza? | No. Y viceversa. Cada eje necesita su propia restricción |
| ¿Qué pasa en las intersecciones? | Las mujeres no blancas son el grupo más excluido; mitigar un solo eje puede perjudicar a otro grupo desfavorecido |
| ¿Hay una solución técnica? | La restricción interseccional (sexo × raza) cumple 4/5 en los tres ejes con ≈1.7 puntos menos de accuracy |
| ¿Cuál es su límite? | Grupos pequeños → métricas ruidosas; equalized odds interseccional poco fiable; categorías agregadas ocultan diferencias |
| ¿Y la edad? | El modelo perjudica a los jóvenes, no a los mayores de 40. Es una disparidad en parte justificable, que debe documentarse |

# 11. Conclusiones

1. **El sesgo existe aunque se oculte el atributo sensible.** El baseline sin `sex` ni `race` selecciona a los
   hombres unas 3 veces más que a las mujeres (DPR 0.30) y a las personas no blancas a la mitad de la tasa de las
   blancas (DPR ≈ 0.56), por los proxies y por el sesgo histórico de la etiqueta.
2. **Exponentiated Gradient funciona como promete el paper.** En test, EG-DP baja la DPD por sexo de 0.18 a 0.03
   (DPR de 0.30 a 0.81) y EG-EO baja la EOD de 0.085 a 0.049, con 1.0 y 0.6 puntos menos de accuracy. La cota
   `difference_bound` controla el trade-off de forma explícita. El resultado es estable frente a `eps`, la semilla y
   el clasificador base, siempre que `max_iter` permita converger.
3. **No hay una sola "equidad".** Optimizar paridad demográfica empeora equalized odds y viceversa, porque las
   tasas base difieren. Elegir la restricción es una **decisión normativa**.
4. **La discriminación es multidimensional.** Mitigar un eje no corrige otro, y puede trasladar la desventaja a
   otro grupo (*fairness gerrymandering*). La restricción **interseccional** es la única que cumple la regla de los
   4/5 en sexo, raza y su combinación a la vez.
5. **Costos y límites:** decisiones aleatorizadas para algunas personas, métricas poco fiables en grupos pequeños,
   categorías agregadas que ocultan diferencias, y la edad como disparidad que requiere una justificación explícita.
6. **Mitigar es necesario pero no suficiente:** el modelo sigue requiriendo datos representativos, documentación,
   supervisión humana y monitoreo, como exige el EU AI Act para sistemas de alto riesgo. Ver el informe escrito.

In [ ]:
# Resúmenes finales exportados para el informe y la presentación
resumen = tabla_comp.drop(columns="usa_sexo_al_predecir").astype(float).round(4)
resumen.to_csv("resumen_modelos_test.csv")
aud_raza5.to_csv("auditoria_raza5_baseline.csv")
sel_inter.to_csv("seleccion_interseccional.csv")
aud_edad.to_csv("auditoria_edad.csv")
ic.to_csv("intervalos_confianza.csv", index=False)
print("Cotas elegidas (sexo):", seleccion)
cruce.round(3)